In [1]:
import sys, os

!pip install -q gymnasium[atari]
!pip install -q tensorboardX
!pip install -q opencv-python
!pip install shimmy

Defaulting to user installation because normal site-packages is not writeable


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
# import tensorflow as torch
# import torch as tf

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

class NatureDQNModel(nn.Module):
    """Same architecture as Nature DQN paper but with two output heads"""
    def __init__(self, n_actions):
        super(NatureDQNModel, self).__init__()

        self.conv1 = nn.Conv2d(4, 32, kernel_size=8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)

        self.fc = nn.Linear(64 * 7 * 7, 512)

        self.logits_head = nn.Linear(512, n_actions)
        self.value_head = nn.Linear(512, 1)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        nn.init.orthogonal_(self.logits_head.weight, gain=0.01)
        nn.init.zeros_(self.logits_head.bias)
        nn.init.orthogonal_(self.value_head.weight, gain=1.0)
        nn.init.zeros_(self.value_head.bias)

    def forward(self, x):
        x = x / 255.0  # Normalize to [0, 1]

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc(x))

        logits = self.logits_head(x)
        values = self.value_head(x)

        return logits, values

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [4]:
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ['actions', 'logits', 'log_probs', 'values'].

        if not isinstance(inputs, torch.Tensor):
            inputs_tensor = torch.FloatTensor(inputs)
        else:
            inputs_tensor = inputs

        with torch.no_grad():
            logits, values = self.model(inputs_tensor)

            probs = F.softmax(logits, dim=-1)
            dist = Categorical(probs)

            actions = dist.sample()
            log_probs = dist.log_prob(actions)

        return {
            'actions': actions.cpu().numpy(),
            'logits': logits.cpu().numpy(),
            'log_probs': log_probs.cpu().numpy(),
            'values': values.cpu().numpy().squeeze(-1)
        }


Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [5]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* 'observations'
* 'rewards'
* 'resets'
* 'actions'
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it's different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t'=0}^{T - 1} \gamma^{t'}r_{t+t'} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory['resets']` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory['state']['latest_observation']`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [8]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        # This method should modify trajectory inplace by adding
        # an item with key 'value_targets' to it.

        rewards = np.array(trajectory['rewards'])
        resets = np.array(trajectory['resets'])
        last_obs = trajectory['state']['latest_observation']

        with torch.no_grad():
            last_obs_tensor = torch.FloatTensor(last_obs)
            _, last_values = self.policy.model(last_obs_tensor)
            last_values = last_values.cpu().numpy().squeeze(-1)

        T = len(rewards)
        nenvs = rewards.shape[1] if rewards.ndim > 1 else 1

        value_targets = np.zeros((T, nenvs))

        next_value = last_values.copy()

        for t in reversed(range(T)):
            bootstrap = 1.0 - resets[t]
            next_value = self.gamma * next_value * bootstrap
            value_targets[t] = rewards[t] + next_value
            next_value = value_targets[t]

        trajectory['value_targets'] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [9]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        # Modify trajectory inplace.

        for key, value in trajectory.items():
            if key == 'state':
                continue

            if isinstance(value, list):
                if len(value) > 0 and isinstance(value[0], np.ndarray):
                    stacked = np.stack(value)  # Shape: (T, nenvs, ...)
                    if stacked.ndim >= 2:
                        new_shape = (-1,) + stacked.shape[2:]
                        trajectory[key] = stacked.reshape(new_shape)
                    else:
                        trajectory[key] = stacked
                else:
                    arr = np.array(value)
                    trajectory[key] = arr.flatten()
            elif isinstance(value, np.ndarray):
                if value.ndim >= 2:
                    new_shape = (-1,) + value.shape[2:]
                    trajectory[key] = value.reshape(new_shape)
                else:
                    trajectory[key] = value.flatten()


In [10]:
model = NatureDQNModel(env.action_space.n)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [13]:
import torch
import torch.nn.functional as F

class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm

    def loss(self, trajectory):
        # Get observations and recompute model outputs to have gradients
        observations = torch.FloatTensor(trajectory['observations'])
        
        # Forward pass (this creates gradients)
        logits, values = self.policy.model(observations)
        
        # Get saved data
        saved_log_probs = torch.FloatTensor(trajectory['log_probs']).squeeze()
        value_targets = torch.FloatTensor(trajectory['value_targets']).squeeze()
        
        # Reshape to match
        values = values.squeeze()
        
        # Compute advantages using current values (detached to avoid double gradient)
        advantages = value_targets - values.detach()
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Policy loss using saved log probs (these don't have gradients, which is fine
        # because we want to treat the actions as fixed)
        p_loss = -(saved_log_probs * advantages).mean()
        
        # Value loss (this will have gradients through values)
        v_loss = F.mse_loss(values, value_targets)
        
        # Entropy bonus (this will have gradients through logits)
        probs = F.softmax(logits, dim=-1)
        log_probs_ent = F.log_softmax(logits, dim=-1)
        ent = -(probs * log_probs_ent).sum(dim=-1).mean()
        
        # Store for logging
        self.last_policy_loss = p_loss.item()
        self.last_value_loss = v_loss.item()
        self.last_entropy = ent.item()
        
        # Total loss
        total_loss = p_loss - self.entropy_coef * ent + self.value_loss_coef * v_loss
        
        return total_loss

    def step(self, trajectory):
        # Compute loss
        loss = self.loss(trajectory)
        
        # Backward pass
        self.optimizer.zero_grad()
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(self.policy.model.parameters(), self.max_grad_norm)
        
        # Calculate gradient norm for logging
        total_norm = 0
        for p in self.policy.model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2).item()
                total_norm += param_norm ** 2
        self.last_grad_norm = total_norm ** 0.5
        
        # Update parameters
        self.optimizer.step()
        
        return loss.item()

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [ ]:
#if you use TensorboardSummaries
%load_ext tensorboard
%tensorboard --logdir logs

In [14]:
from torch.optim import RMSprop
from collections import deque
import time

initial_lr = 7e-4
total_steps = 5_000_000
optimizer = RMSprop(model.parameters(), lr=initial_lr, alpha=0.99, eps=1e-5)

a2c = A2C(
    policy=policy,
    optimizer=optimizer,
    value_loss_coef=0.5,
    entropy_coef=0.01,
    max_grad_norm=0.5
)

episode_rewards = deque(maxlen=100)
current_episode_rewards = np.zeros(nenvs)
episode_lengths = deque(maxlen=100)

total_interactions = 0
max_interactions = total_steps
log_interval = 1000
save_interval = 100000

print("Starting training...")
start_time = time.time()

while total_interactions < max_interactions:
    trajectory = runner.get_next()

    total_interactions += len(trajectory['rewards'])

    for i, (reward, reset) in enumerate(zip(trajectory['rewards'][-nenvs:], trajectory['resets'][-nenvs:])):
        current_episode_rewards[i] += reward
        if reset:
            episode_rewards.append(current_episode_rewards[i])
            current_episode_rewards[i] = 0

    loss = a2c.step(trajectory)

    progress = total_interactions / max_interactions
    current_lr = initial_lr * (1 - progress)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    if total_interactions % log_interval == 0:
        avg_reward = np.mean(episode_rewards) if episode_rewards else 0
        elapsed_time = time.time() - start_time
        steps_per_sec = total_interactions / elapsed_time

        print(f"Steps: {total_interactions:8d}/{max_interactions} | "
              f"Avg Reward (100 ep): {avg_reward:6.1f} | "
              f"Loss: {loss:6.4f} | "
              f"Policy Loss: {a2c.last_policy_loss:6.4f} | "
              f"Value Loss: {a2c.last_value_loss:6.4f} | "
              f"Entropy: {a2c.last_entropy:6.4f} | "
              f"Grad Norm: {a2c.last_grad_norm:6.4f} | "
              f"LR: {current_lr:.6f} | "
              f"FPS: {steps_per_sec:.1f}")

    if total_interactions % save_interval == 0:
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'total_interactions': total_interactions,
            'avg_reward': np.mean(episode_rewards) if episode_rewards else 0
        }, f'a2c_checkpoint_{total_interactions}.pt')
        print(f"Checkpoint saved at {total_interactions} steps")

print("Training completed!")

Starting training...
Steps:     1000/5000000 | Avg Reward (100 ep):    0.0 | Loss: 0.1479 | Policy Loss: -0.0000 | Value Loss: 0.3317 | Entropy: 1.7918 | Grad Norm: 0.5000 | LR: 0.000700 | FPS: 51.0
Steps:     2000/5000000 | Avg Reward (100 ep):    0.2 | Loss: -0.0178 | Policy Loss: 0.0000 | Value Loss: 0.0001 | Entropy: 1.7918 | Grad Norm: 0.0585 | LR: 0.000700 | FPS: 57.2
Steps:     3000/5000000 | Avg Reward (100 ep):    0.3 | Loss: -0.0172 | Policy Loss: -0.0000 | Value Loss: 0.0013 | Entropy: 1.7918 | Grad Norm: 0.4736 | LR: 0.000700 | FPS: 62.1
Steps:     4000/5000000 | Avg Reward (100 ep):    0.3 | Loss: 0.2810 | Policy Loss: 0.0000 | Value Loss: 0.5979 | Entropy: 1.7918 | Grad Norm: 0.5000 | LR: 0.000699 | FPS: 66.2
Steps:     5000/5000000 | Avg Reward (100 ep):    0.8 | Loss: -0.0175 | Policy Loss: -0.0000 | Value Loss: 0.0008 | Entropy: 1.7918 | Grad Norm: 0.2740 | LR: 0.000699 | FPS: 70.5
Steps:     6000/5000000 | Avg Reward (100 ep):    0.9 | Loss: -0.0177 | Policy Loss: -0.

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.